# v15 · Breaking it: Kasiski, index of coincidence, per-column Caesar solve
`PtolemyDesktop/Kryptos/analysis.py`. The same tools attack the PRIVATE-charset construction (v10).

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
import sys
sys.path.insert(0, os.path.join(PROJ, "PtolemyDesktop"))
vg = load("PtolemyDesktop/Kryptos/Ciphers/Vigenere.py", "vg")
an = load("PtolemyDesktop/Kryptos/analysis.py", "an")

## Vigenère on the corpus letters

In [3]:
plain = an.clean(CORPUS)
print("plaintext letters:", len(plain), "| IoC:", round(an.index_of_coincidence(plain), 4))
for key in ["LEMON", "HYPERINDEX"]:
    ct = vg.encrypt(plain, key)
    res = an.break_vigenere(ct)
    print("key %-11s -> ciphertext IoC %.4f | recovered period %d, key %s, plaintext exact: %s" % (key, an.index_of_coincidence(ct), res["period"], res["key"], res["plaintext"] == plain))

plaintext letters: 3085 | IoC: 0.0636
key LEMON       -> ciphertext IoC 0.0412 | recovered period 5, key LEMON, plaintext exact: True
key HYPERINDEX  -> ciphertext IoC 0.0407 | recovered period 2, key RE, plaintext exact: False


## When the key length has small factors, the breaker's period vote picks the wrong one
For `HYPERINDEX` (length 10) the recovered period is 2: every gap between repeated trigrams is a multiple of 10, hence also of 2, and `break_vigenere` takes the top-voted factor. Given the true period the per-column solve is exact.

In [4]:
ct = vg.encrypt(plain, "HYPERINDEX")
votes, _ = an.kasiski(ct)
print("Kasiski votes (factor, votes):", votes[:6])
print("key from true period 10:", an.vigenere_key_from_period(ct, 10))
assert an.vigenere_key_from_period(ct, 10) == "HYPERINDEX"

Kasiski votes (factor, votes): [(2, 2017), (5, 1985), (10, 1950), (4, 595), (20, 562), (3, 447)]
key from true period 10: HYPERINDEX


## The PRIVATE charset (v10) is a monoalphabetic substitution on the address digits
A private address is the public address of the substituted text. A reader who knows only the base reads the substituted text off the integer. Substitution preserves the *repeat pattern* of every word — the key of al-Kindi's pattern-dictionary attack.

In [5]:
import random, re
hw2 = load("PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/hyperwebster.py", "hw2")
pub = hw2.HyperWebster(); base = pub.characters
rnd = random.Random(2026); perm = list(base); rnd.shuffle(perm)
priv = hw2.HyperWebster("".join(perm))
text = CORPUS[:3000]
seen = pub.regenerate_text(priv.point_to_text(text))          # what an observer reconstructs from the integer
assert len(seen) == len(text)
def pattern(w):
    m, out = {}, []
    for c in w: out.append(m.setdefault(c, len(m) + 1))
    return tuple(out)
spans = [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]
same = sum(pattern(text[a:b]) == pattern(seen[a:b]) for a, b in spans)
print("words:", len(spans), "| repeat pattern identical after substitution:", same)
assert same == len(spans)
# how much does the pattern alone narrow a word? candidate set = dictionary words with the same pattern
dictionary = sorted({text[a:b].lower() for a, b in spans if b - a >= 4})
by_pat = {}
for w in dictionary: by_pat.setdefault(pattern(w), []).append(w)
unique = sum(len(v) == 1 for v in by_pat.values())
print("dictionary of", len(dictionary), "distinct words (length >= 4):", len(by_pat), "distinct patterns;", unique, "patterns pick out a single word")

words: 304 | repeat pattern identical after substitution: 304
dictionary of 167 distinct words (length >= 4): 109 distinct patterns; 97 patterns pick out a single word


The word boundaries here are taken from the plaintext; an observer would locate them from the substituted space symbol, which is one more symbol to identify. What the numbers show is the general fact: **a per-symbol relabelling of the digits leaves the pattern structure of the text intact**, which is what the pattern-dictionary attack uses.